# RootScope: Cross-species Root Cell-Type Classification from Confocal Microscopy Images

RootScope uses **Cellpose-SAM** to segment every cell and describes each one
with morpho-topological features and fine-tuned **DINOv2** embeddings.
Classification is refined through an iterative process in which predictions
from neighboring cells are fed back as new features, updating every label
until the classification converges. Each cell is then assigned to one of nine
anatomical types.

- Code: https://github.com/ct-tranchau/Rootscope
- Weights: https://huggingface.co/ct-tranchau/Rootscope

### Before you start: turn on the GPU

In [ ]:
#@title 1. Check the GPU is on { display-mode: "form" }
import subprocess, sys
try:
    print(subprocess.check_output(["nvidia-smi",
        "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip())
except Exception:
    print("NO GPU DETECTED.")
    print("Runtime > Change runtime type > T4 GPU, then re-run this cell.")
print("Python", sys.version.split()[0])

In [ ]:
#@title 2. Install (~3-5 min, restart when prompted) { display-mode: "form" }
%pip install -q "numpy<2.1" "git+https://github.com/ct-tranchau/Rootscope.git"
print("\nInstall finished.")
print("If Colab shows a RESTART SESSION button, click it, then continue at step 3.")


## 3. Load the models

First run downloads ~460 MB of classifiers from Hugging Face, the ~85 MB
fine-tuned DINOv2 backbone, and the Cellpose-SAM checkpoint. They are cached,
so later runs in the same session start instantly.

In [ ]:
#@title 3. Load models onto the GPU { display-mode: "form" }
import torch
from rootscope import predict as rs
from rootscope.cnn_embeddings import load_dinov2
from rootscope.extract_features import load_cellpose_model
from rootscope.weights import resolve_cnn_weights, resolve_model_dir

GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "NONE - this will be slow")

MODEL_DIR   = resolve_model_dir()
CNN_WEIGHTS = resolve_cnn_weights()
MODELS, SCALERS, FEATURE_COLS, LE = rs.load_models(str(MODEL_DIR))
CELLPOSE = load_cellpose_model(use_gpu=GPU)
DINOV2   = load_dinov2(weights_path=str(CNN_WEIGHTS) if CNN_WEIGHTS else None,
                       use_gpu=GPU)
print(f"\nReady: {len(MODELS)} classifiers, {len(FEATURE_COLS)} features")
print("Classes:", list(LE.classes_))

## 4. Upload your TIFF

To use your own image, untick `USE_EXAMPLE` in the cell below, then run it and
pick the file. Leave it ticked to run the bundled example.

In [ ]:
#@title 4. Upload a TIFF (or skip to use an example) { display-mode: "form" }
import re, tifffile
from pathlib import Path
from google.colab import files

USE_EXAMPLE = True  #@param {type:"boolean"}

if USE_EXAMPLE:
    import urllib.request
    TIF = "Spennellii_RootTip_EarlyMaturation.tif"
    if not Path(TIF).exists():
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/ct-tranchau/Rootscope/main/"
            "examples/Spennellii_RootTip_EarlyMaturation.tif", TIF)
    print("Using bundled example:", TIF)
else:
    uploaded = files.upload()
    TIF = list(uploaded.keys())[0]
    print("Uploaded:", TIF)

def read_um_per_px(path):
    try:
        with tifffile.TiffFile(path) as tf:
            if tf.ome_metadata:
                m = re.search(r'PhysicalSizeX="([0-9.eE+-]+)"', tf.ome_metadata)
                if m:
                    return round(float(m.group(1)), 6)
    except Exception as e:
        print("  could not read metadata:", e)
    return None

detected = read_um_per_px(TIF)
if detected:
    UM_PER_PX = detected
    print(f"Pixel size from metadata: {UM_PER_PX} um/px")
else:
    UM_PER_PX = 1.0
    print("NO pixel size in metadata. Set UM_PER_PX by hand in the next cell,")
    print("otherwise every size-derived feature will be wrong.")

In [ ]:
#@title 4b. Override the pixel size if you need to { display-mode: "form" }
OVERRIDE = False   #@param {type:"boolean"}
VALUE    = 1.0     #@param {type:"number"}
if OVERRIDE:
    UM_PER_PX = VALUE
print("Using UM_PER_PX =", UM_PER_PX)

## 5. Run

Segmentation and the DINOv2 embeddings run on the GPU; feature extraction and
the iterative ensemble run on the CPU.

In [ ]:
#@title 5. Segment + classify { display-mode: "form" }
import time
from pathlib import Path

OUT = Path("results"); OUT.mkdir(exist_ok=True)
stem = Path(TIF).stem
t0 = time.time()

img = rs.load_image(TIF)
print("Image:", img.shape)

masks = rs.stage_segment(img, gpu=GPU, cellpose_model=CELLPOSE)
print(f"Segmented: {int(masks.max())} cells  ({time.time()-t0:.0f}s)")

masks, df_base, layers, adjacency, n_layers = rs.stage_features(
    masks, img, um_per_px=UM_PER_PX)
df_base = rs.stage_embed(masks, img, df_base, gpu=GPU, dinov2_model=DINOV2)

df = rs.stage_classify(
    df_base, masks, img, layers, adjacency,
    MODELS, SCALERS, FEATURE_COLS, LE,
    out_dir=OUT, stem=stem, source_name=Path(TIF).name,
    um_per_px=UM_PER_PX, max_rounds=10, label_cells=False)

print(f"\nDone in {time.time()-t0:.0f}s -> {OUT}/")

## 6. Look at the result

In [ ]:
#@title 6. Show the overlay and per-type counts { display-mode: "form" }
import matplotlib.pyplot as plt
from PIL import Image

MODEL = "Ensemble"  #@param ["Ensemble", "RandomForest", "LightGBM", "XGBoost"]

overlay = OUT / f"{stem}_{MODEL}_overlay.png"
plt.figure(figsize=(11, 11))
plt.imshow(Image.open(overlay)); plt.axis("off"); plt.title(f"{stem} - {MODEL}")
plt.show()

sub = df[df["model"] == MODEL]
counts = sub["predicted_cell_type"].value_counts()
summary = counts.to_frame("cells")
summary["% of cells"] = (100 * counts / counts.sum()).round(1)
summary["mean confidence"] = [
    round(sub.loc[sub["predicted_cell_type"] == ct, "prediction_confidence"].mean(), 3)
    for ct in counts.index]
print(f"{len(sub)} cells, {n_layers} tissue layers, {UM_PER_PX} um/px")
print(f"xylem poles: {int(sub.iloc[0]['n_xylem_poles'])}, "
      f"phloem poles: {int(sub.iloc[0]['n_phloem_poles'])}\n")
display(summary)

## 7. Download the results

In [ ]:
#@title 7. Download everything as a ZIP { display-mode: "form" }
import shutil
from google.colab import files
shutil.make_archive(f"{stem}_rootscope", "zip", OUT)
files.download(f"{stem}_rootscope.zip")

---

## Batch: a folder of images from Google Drive

Put your TIFFs in a Drive folder and point `TIF_DIR` at it. Results are
written back to Drive.

In [ ]:
#@title Batch over a Drive folder { display-mode: "form" }
RUN_BATCH = False  #@param {type:"boolean"}

if RUN_BATCH:
    from google.colab import drive
    import pandas as pd
    drive.mount("/content/drive")

    TIF_DIR     = "/content/drive/MyDrive/root_tifs"      #@param {type:"string"}
    OUT_DIR     = "/content/drive/MyDrive/root_results"   #@param {type:"string"}
    BATCH_UM_PX = 0.4546                                  #@param {type:"number"}

    out = Path(OUT_DIR); out.mkdir(parents=True, exist_ok=True)
    tifs = sorted(Path(TIF_DIR).glob("*.tif"))
    print(f"{len(tifs)} images found\n")

    tables = []
    for i, tp in enumerate(tifs, 1):
        print(f"[{i}/{len(tifs)}] {tp.name}")
        try:
            im = rs.load_image(tp)
            mk = rs.stage_segment(im, gpu=GPU, cellpose_model=CELLPOSE)
            if int(mk.max()) == 0:
                print("   no cells, skipped"); continue
            mk, base, ll, adj, nl = rs.stage_features(mk, im, um_per_px=BATCH_UM_PX)
            if base is None:
                print("   all debris, skipped"); continue
            base = rs.stage_embed(mk, im, base, gpu=GPU, dinov2_model=DINOV2)
            d = rs.stage_classify(base, mk, im, ll, adj,
                                  MODELS, SCALERS, FEATURE_COLS, LE,
                                  out_dir=out, stem=tp.stem, source_name=tp.name,
                                  um_per_px=BATCH_UM_PX, max_rounds=10)
            tables.append(d)
        except Exception as e:
            print(f"   FAILED: {e}")

    if tables:
        combined = pd.concat(tables, ignore_index=True)
        combined.to_csv(out / "all_predictions.csv", index=False)
        print(f"\n{len(combined)} rows -> {out}/all_predictions.csv")